Imports / Configurações

In [5]:
import pandas as pd
import numpy as np
import pickle
from collections import Counter
import math
import sys

sys.modules['src'] = sys.modules['__main__']
sys.modules['src.text_processing'] = sys.modules['__main__']
sys.modules['src.numpy_models'] = sys.modules['__main__']

Definir Funções

In [6]:
class CustomTFIDF:
    def __init__(self, max_features=1000):

        self.max_features = max_features
        self.vocab = {}
        self.idf = {}

    def _tokenize(self, text):

        import re
        text = str(text).lower()
        text = re.sub(r"[^a-z0-9\s]", "", text)
        return text.split()

    def fit(self, texts):

        df = Counter() # Document Frequency
        N = len(texts)
        
        for text in texts:
            words = set(self._tokenize(text)) # set -> remover dups
            for w in words:
                df[w] += 1
                
        top_words = df.most_common(self.max_features)
        self.vocab = {word: i for i, (word, count) in enumerate(top_words)}
        
            # log(N / (1 + docs com palavra))
        self.idf = {word: math.log(N / (1 + df[word])) for word in self.vocab}
        print(f"Vocabulário criado com {len(self.vocab)} palavras")

    def transform(self, texts):

        X = np.zeros((len(texts), len(self.vocab)))
        
        for i, text in enumerate(texts):
            words = self._tokenize(text)
            word_counts = Counter(words)
            total_words = len(words)
            
            if total_words == 0:
                continue
                
            for word, count in word_counts.items():
                if word in self.vocab:
                        
                    tf = count / float(total_words)
                    col_idx = self.vocab[word]
                    X[i, col_idx] = tf * self.idf[word]
                    
        return X

class NumpyDNN:
    def __init__(self, input_size, hidden_size, output_size, learning_rate=0.01):
        self.lr = learning_rate
        
        self.W1 = np.random.randn(input_size, hidden_size) * 0.01
        self.b1 = np.zeros((1, hidden_size))
        
        self.W2 = np.random.randn(hidden_size, output_size) * 0.01
        self.b2 = np.zeros((1, output_size))

    def relu(self, Z):
        return np.maximum(0, Z)

    def relu_derivative(self, Z):
        return Z > 0

    def softmax(self, Z):
        expZ = np.exp(Z - np.max(Z, axis=1, keepdims=True))
        return expZ / np.sum(expZ, axis=1, keepdims=True)

    def compute_loss(self, Y_hat, Y):       # Categorical Cross-Entropy Loss
        m = Y.shape[0]
        epsilon = 1e-15
        loss = -np.sum(Y * np.log(Y_hat + epsilon)) / m
        return loss

    #* PROPAGAÇÃO

    def forward(self, X):
        self.Z1 = np.dot(X, self.W1) + self.b1
        self.A1 = self.relu(self.Z1)
        
        self.Z2 = np.dot(self.A1, self.W2) + self.b2
        self.A2 = self.softmax(self.Z2)
        return self.A2

    def backward(self, X, Y):
        m = X.shape[0]

        dZ2 = self.A2 - Y
        dW2 = np.dot(self.A1.T, dZ2) / m
        db2 = np.sum(dZ2, axis=0, keepdims=True) / m

        dA1 = np.dot(dZ2, self.W2.T)
        dZ1 = dA1 * self.relu_derivative(self.Z1)
        dW1 = np.dot(X.T, dZ1) / m
        db1 = np.sum(dZ1, axis=0, keepdims=True) / m

        self.W1 -= self.lr * dW1
        self.b1 -= self.lr * db1
        self.W2 -= self.lr * dW2
        self.b2 -= self.lr * db2

    #* TREINO E PREVISÃO
    def train(self, X, Y, epochs=1000):
        print("A iniciar o treino da rede neuronal")
        for i in range(epochs):
            Y_hat = self.forward(X)
            loss = self.compute_loss(Y_hat, Y)
            self.backward(X, Y)
            
            if i % 100 == 0 or i == epochs - 1:

                previsoes = np.argmax(Y_hat, axis=1)
                reais = np.argmax(Y, axis=1)
                accuracy = np.mean(previsoes == reais) * 100

                print(f"Época {i} | Loss: {loss:.4f} | Accuracy: {accuracy:.2f}%")
        print("Treino concluído")

    def predict(self, X):
        probs = self.forward(X)

        return np.argmax(probs, axis=1)

Carregar Modelo

In [7]:
with open('../modelos/modelo_A.pkl', 'rb') as f:
    dados_guardados = pickle.load(f)

tfidf_treinado = dados_guardados['tfidf']
modelo_treinado = dados_guardados['modelo']
reverse_map = dados_guardados['reverse_map']

df_subm = pd.read_csv("../data/subm1.csv", sep=";")

X_subm = tfidf_treinado.transform(df_subm['Text'].values)
previsoes_idx = modelo_treinado.predict(X_subm)
previsoes_labels = [reverse_map[idx] for idx in previsoes_idx]

df_subm['Labels'] = previsoes_labels

nome_saida = "subm1-g1-MEI-A.csv"
df_subm.to_csv(nome_saida, index=False)